In [ ]:
# ==========================================================
# INSTALLATION: Run this in your terminal or uncomment with '!' in Jupyter
# !pip install crewai langchain-openai pypdf fpdf
# ==========================================================

import os
import signal
import sys
import json
import re
import time
import PyPDF2
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import BaseTool
from pydantic import BaseModel, Field
from fpdf import FPDF

# ==========================================================
# 0. WINDOWS COMPATIBILITY PATCH (CRITICAL)
# ==========================================================
valid_signals = {'SIGHUP': 1, 'SIGTSTP': 2, 'SIGCONT': 3, 'SIGQUIT': 3, 'SIGTRAP': 5}
for sig, val in valid_signals.items():
    if not hasattr(signal, sig):
        setattr(signal, sig, val)

# ==========================================================
# 1. USER CONFIGURATION
# ==========================================================
# 🛑 NEVER commit your actual API key to GitHub. 
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE" 

# Uses a relative path. Ensure your PDF is in a folder named 'data' next to this script.
PDF_FILE_PATH = "data/tsla-20241231.pdf"
JSON_OUTPUT_FILE = "extracted_financial_data.json"

my_llm = LLM(
    model="openai/gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0.0
)

# ==========================================================
# 2. TOOLS
# ==========================================================

class PDFToolInput(BaseModel):
    file_path: str = Field(..., description="Path to the PDF file.")

class PDFReadTool(BaseTool):
    name: str = "Read PDF Tool"
    description: str = "Reads the first 35 pages of a PDF to extract financial text."
    args_schema: type[BaseModel] = PDFToolInput

    def _run(self, file_path: str) -> str:
        try:
            if not os.path.exists(file_path): return "Error: File not found."
            with open(file_path, 'rb') as file:
                reader = PyPDF2.PdfReader(file)
                text = ""
                for i in range(min(len(reader.pages), 35)):
                    text += reader.pages[i].extract_text() + "\n"
            return text
        except Exception as e: return f"Error: {e}"

class SaveJSONInput(BaseModel):
    json_data: str = Field(..., description="The valid JSON string to save.")

class SaveJSONTool(BaseTool):
    name: str = "Save Financial Data"
    description: str = "Saves extracted data to a local JSON file."
    args_schema: type[BaseModel] = SaveJSONInput

    def _run(self, json_data: str) -> str:
        try:
            clean_str = json_data.replace('```json', '').replace('```', '').strip()
            data = json.loads(clean_str)
            if "balance_sheet" not in data or "income_statement" not in data:
                return "Error: JSON missing 'balance_sheet' or 'income_statement'."
            with open(JSON_OUTPUT_FILE, 'w') as f:
                json.dump(data, f, indent=4)
            return f"SUCCESS: Data saved to {JSON_OUTPUT_FILE}."
        except Exception as e: return f"Error saving JSON: {e}"

class RatioInput(BaseModel):
    pass 

class CreditRatioTool(BaseTool):
    name: str = "Credit Ratio Calculator"
    description: str = f"Calculates ratios from {JSON_OUTPUT_FILE} and saves them for the report."
    args_schema: type[BaseModel] = RatioInput

    def _run(self) -> str:
        try:
            if not os.path.exists(JSON_OUTPUT_FILE): return "Error: Data file missing."
            with open(JSON_OUTPUT_FILE, 'r') as f:
                data = json.load(f)

            bs = data.get('balance_sheet', {})
            is_ = data.get('income_statement', {})
            years = sorted(bs.get('total_assets', {}).keys(), reverse=True)
            
            ratios_output = {
                "Current Ratio": {},
                "Debt-to-Equity": {},
                "Interest Coverage Ratio (x)": {},
                "Net Margin (%)": {}
            }

            for year in years:
                def get_val(dataset, metric, yr):
                    val = dataset.get(metric, {}).get(yr, 0)
                    if isinstance(val, str): 
                        val = float(val.replace(',', '').replace('$', '').replace(' ', ''))
                    return float(val)

                curr_assets = get_val(bs, 'total_current_assets', year)
                curr_liab = get_val(bs, 'total_current_liabilities', year)
                curr_ratio = round(curr_assets / curr_liab, 2) if curr_liab else 0

                tot_liab = get_val(bs, 'total_liabilities', year)
                tot_equity = get_val(bs, 'total_equity', year)
                d_e = round(tot_liab / tot_equity, 2) if tot_equity else 0

                op_income = get_val(is_, 'operating_income', year)
                int_exp = abs(get_val(is_, 'interest_expense', year))
                int_cov = round(op_income / int_exp, 2) if int_exp else 999.99 

                net_inc = get_val(is_, 'net_income', year)
                rev = get_val(is_, 'total_revenue', year)
                margin = round((net_inc / rev) * 100, 2) if rev else 0

                ratios_output["Current Ratio"][year] = curr_ratio
                ratios_output["Debt-to-Equity"][year] = d_e
                ratios_output["Interest Coverage Ratio (x)"][year] = int_cov
                ratios_output["Net Margin (%)"][year] = margin

            data['financial_ratios'] = ratios_output
            with open(JSON_OUTPUT_FILE, 'w') as f:
                json.dump(data, f, indent=4)

            return json.dumps(ratios_output, indent=2)
        except Exception as e: return f"Error calculating ratios: {e}"

class PDFInput(BaseModel):
    analysis_text: str = Field(..., description="The Analyst's text summary.")

class PDFReportTool(BaseTool):
    name: str = "PDF Report Creator"
    description: str = "Generates the final PDF."
    args_schema: type[BaseModel] = PDFInput

    def _run(self, analysis_text: str) -> str:
        try:
            if not os.path.exists(JSON_OUTPUT_FILE): return "Error: Data file missing."
            with open(JSON_OUTPUT_FILE, 'r') as f:
                data = json.load(f)

            pdf = FPDF()
            pdf.add_page()
            
            pdf.set_font("Arial", 'B', 16)
            pdf.cell(0, 10, "Credit Portfolio Report", ln=True, align='C')
            pdf.set_font("Arial", 'I', 10)
            pdf.cell(0, 10, "Generated by AI Credit Agent", ln=True, align='C')
            pdf.ln(5)

            pdf.set_font("Arial", 'B', 12)
            pdf.set_fill_color(220, 230, 241) 
            pdf.cell(0, 10, "  1. Executive Credit Summary", ln=True, fill=True)
            pdf.ln(2)
            
            # === MODIFICATION START: Removing Stars ===
            clean_text = analysis_text.replace('**', '').replace('__', '')
            # ==========================================
            
            pdf.set_font("Arial", '', 11)
            safe_text = clean_text.encode('latin-1', 'replace').decode('latin-1')
            pdf.multi_cell(0, 7, safe_text.strip())
            pdf.ln(5)

            pdf.set_font("Arial", 'B', 12)
            pdf.cell(0, 10, "  2. Financial Statements & Ratios", ln=True, fill=True)
            pdf.ln(5)

            bs = data.get('balance_sheet', {})
            years = sorted(bs.get('total_assets', {}).keys(), reverse=True)
            
            label_w = 70
            col_w = min(40, (190 - label_w) / max(1, len(years))) 

            pdf.set_font("Arial", 'B', 9)
            pdf.cell(label_w, 10, "Line Item", 1)
            for year in years:
                pdf.cell(col_w, 10, str(year), 1, align='C')
            pdf.ln()

            pdf.set_font("Arial", '', 9)
            def add_section(data_dict, section_name):
                if not data_dict: return
                pdf.set_font("Arial", 'B', 9)
                pdf.cell(0, 8, f"  {section_name}", ln=True)
                pdf.set_font("Arial", '', 9)
                for k, year_data in data_dict.items():
                    clean_k = k.replace('_', ' ').title()
                    pdf.cell(label_w, 8, clean_k, 1)
                    for year in years:
                        val = year_data.get(year, 0)
                        if section_name == "Key Financial Ratios":
                            fmt_val = f"{val:,.2f}"
                        else:
                            fmt_val = f"{val:,.0f}"
                        pdf.cell(col_w, 8, fmt_val, 1, align='C')
                    pdf.ln()

            add_section(data.get('balance_sheet', {}), "Balance Sheet (USD Millions)")
            add_section(data.get('income_statement', {}), "Income Statement (USD Millions)")
            add_section(data.get('financial_ratios', {}), "Key Financial Ratios")

            output_file = "Final_Credit_Report.pdf"
            pdf.output(output_file)
            return f"SUCCESS: Report saved as {output_file}"

        except Exception as e: return f"Error creating PDF: {e}"

# ==========================================================
# 3. AGENTS
# ==========================================================

pdf_tool = PDFReadTool()
save_tool = SaveJSONTool()
ratio_tool = CreditRatioTool()
report_tool = PDFReportTool()

extractor_agent = Agent(
    role='Data Extractor',
    goal='Extract financial data exactly as it appears in the PDF.',
    backstory="You are a meticulous data auditor. You map columns strictly: Left=Newest, Right=Oldest.",
    tools=[pdf_tool, save_tool],
    llm=my_llm,
    verbose=True, memory=False
)

analyst_agent = Agent(
    role='Credit Analyst',
    goal='Analyze financial ratios and write a concise opinion.',
    backstory="You are a credit officer. You rely on the Ratio Calculator tool for insights.",
    tools=[ratio_tool],
    llm=my_llm,
    verbose=True, memory=False
)

reporter_agent = Agent(
    role='Reporting Officer',
    goal='Generate the final PDF document.',
    backstory="You are a reporting specialist.",
    tools=[report_tool],
    llm=my_llm,
    verbose=True, memory=False
)

# ==========================================================
# 4. ORCHESTRATION
# ==========================================================

def run_credit_agent():
    print("\n--- 1. STARTING EXTRACTION ---")
    
    extraction_task = Task(
        description=f"""
        1. Read the PDF at {PDF_FILE_PATH}.
        2. Extract Balance Sheet and Income Statement data for ALL years found.
        
        **CRITICAL SPATIAL RULES:**
        - In financial tables, the **LEFTMOST** number column is the **MOST RECENT** year (e.g., 2024).
        - The **RIGHT** column is the **OLDER** year (e.g., 2023).
        - Example: If row is "Net Income 14,974 7,153", then 2024=14,974 and 2023=7,153.
        
        3. EXTRACT THESE FIELDS:
           - Balance Sheet: total_assets, total_current_assets, total_liabilities, total_current_liabilities, total_equity
           - Income Statement: total_revenue, operating_income, net_income, interest_expense
           
        4. CALL TOOL 'Save Financial Data' to save the valid JSON.
        """,
        expected_output="Confirmation that data was saved.",
        agent=extractor_agent
    )
    
    crew_1 = Crew(agents=[extractor_agent], tasks=[extraction_task])
    result_1 = crew_1.kickoff()
    
    if "Error" in str(result_1):
        print("Extraction failed. Stopping pipeline.")
        return

    time.sleep(2)

    print("\n--- 2. STARTING ANALYSIS ---")
    analysis_task = Task(
        description="""
        1. Call 'Credit Ratio Calculator'.
        2. Compare 2024 vs 2023.
        3. Write a brief professional opinion on Liquidity and Profitability.
        """,
        expected_output="Credit opinion summary.",
        agent=analyst_agent
    )

    crew_2 = Crew(agents=[analyst_agent], tasks=[analysis_task])
    analysis_text = str(crew_2.kickoff())

    print("\n--- 3. GENERATING REPORT ---")
    report_task = Task(
        description=f"""
        1. Call 'PDF Report Creator'.
        2. Pass the analysis text below.
        
        TEXT:
        {analysis_text}
        """,
        expected_output="PDF Success message.",
        agent=reporter_agent
    )

    crew_3 = Crew(agents=[reporter_agent], tasks=[report_task])
    result = crew_3.kickoff()
    print(f"\n--- DONE: {result} ---")

if __name__ == "__main__":
    run_credit_agent()